In [1]:
import pandas as pd
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import mlflow
import mlflow.sklearn
import json
import dagshub
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

In [6]:
pip install dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.4/263.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.8 MB/s eta 0:00:00


In [37]:
mlflow.set_tracking_uri("https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow")

dagshub.init(repo_owner='PriyanshuMewal', repo_name='delivery-time-prediction', mlflow=True)

mlflow.set_experiment("Experiment 6: Meta model selection for stacking regression")

Accessing as PriyanshuMewal

Initialized MLflow to track repo "PriyanshuMewal/delivery-time-prediction"

Repository PriyanshuMewal/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/fad99b2f20c04ae3afdad499e263d8fb', creation_time=1771591722244, experiment_id='8', last_update_time=1771591722244, lifecycle_stage='active', name='Experiment 6: Meta model selection for stacking regression', tags={}, workspace='default'>

In [2]:
train_df = pd.read_csv("/content/train_df.csv")
test_df = pd.read_csv("/content/test_df.csv")

In [3]:
X_train = train_df.drop(columns=["time"])
X_test = test_df.drop(columns=["time"])

y_train = train_df["time"]
y_test = test_df["time"]

In [12]:
X_train.shape

(33472, 26)

In [ ]:
y_test.isnull().sum()

0

### Meta model selection:

In [18]:
with open("/content/best_params.json", "r") as file:
    best_params = json.load(file)

# Create base models:
for model, params in best_params.items():

    if model == "lightgbm":
        lightgbm = LGBMRegressor(**params, verbose=-1)
    elif model == "XGboost":
        xgb = XGBRegressor(**params)

estimators = [("xgboost", xgb), ("lightgbm", lightgbm)]

In [17]:
lightgbm.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.01184348616544254,
 'max_depth': 11,
 'min_child_samples': 23,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 1043,
 'n_jobs': None,
 'num_leaves': 137,
 'objective': None,
 'random_state': None,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'feature_fraction': 0.7839333790779994,
 'bagging_fraction': 0.9697068541421031,
 'lambda_l1': 3.972979937157174e-08,
 'lambda_l2': 2.639574087334251e-08}

In [13]:
best_params['lightgbm'].update({"verbose": -1})
best_params

{'lightgbm': {'n_estimators': 1043,
  'learning_rate': 0.01184348616544254,
  'num_leaves': 137,
  'max_depth': 11,
  'min_child_samples': 23,
  'feature_fraction': 0.7839333790779994,
  'bagging_fraction': 0.9697068541421031,
  'lambda_l1': 3.972979937157174e-08,
  'lambda_l2': 2.639574087334251e-08,
  'verbose': -1},
 'XGboost': {'n_estimators': 503,
  'learning_rate': 0.016063959766841125,
  'max_depth': 10,
  'min_child_weight': 3.6328861966907344,
  'gamma': 0.19546838376630365,
  'subsample': 0.85129326702622,
  'colsample_bytree': 0.9778384789059209,
  'reg_alpha': 0.011603836281718623,
  'reg_lambda': 1.2982059192370237e-05},
 'catboost': {'iterations': 1658,
  'learning_rate': 0.012937919335703926,
  'depth': 10,
  'l2_leaf_reg': 1.6558560841722005,
  'loss_function': 'RMSE',
  'bagging_temperature': 0.9829734460508931,
  'min_data_in_leaf': 64}}

In [19]:
def objective(trial):

    # select the meta estimator:
    meta_estimator = trial.suggest_categorical("final_estimator", ["lr", "dt", "knn"])

    if meta_estimator == "lr":
        final_estimator = LinearRegression()
    elif meta_estimator == "dt":
        max_depth = trial.suggest_int("max_depth", 1, 10)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
        final_estimator = DecisionTreeRegressor(max_depth=max_depth,
                                                min_samples_split=min_samples_split,
                                                min_samples_leaf=min_samples_leaf,
                                                random_state=42)
    elif meta_estimator == "knn":
        n_neighbors = trial.suggest_int("n_neighbors", 1, 15)
        weights = trial.suggest_categorical("weights", ["uniform", "distance"])
        final_estimator = KNeighborsRegressor(n_neighbors=n_neighbors,
                                              weights=weights, n_jobs=-1)

    regressor = StackingRegressor(estimators=estimators,
                                  final_estimator=final_estimator, cv=5)

    regressor.fit(X_train, y_train)

    y_pred = regressor.predict(X_test)

    # calculate metrics:
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    trial.set_user_attr("r2", r2)
    return mae

In [ ]:
# study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

[I 2026-02-20 17:30:19,796] A new study created in memory with name: no-name-5033ce42-71af-470f-9e67-476737981df3
[I 2026-02-20 17:31:50,261] Trial 0 finished with value: 0.5391491804682785 and parameters: {'final_estimator': 'dt', 'max_depth': 1, 'min_samples_split': 9, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5391491804682785.
[I 2026-02-20 17:33:18,905] Trial 1 finished with value: 0.34692963811284144 and parameters: {'final_estimator': 'knn', 'n_neighbors': 13, 'weights': 'distance'}. Best is trial 1 with value: 0.34692963811284144.
[I 2026-02-20 17:34:47,324] Trial 2 finished with value: 0.33066565035252565 and parameters: {'final_estimator': 'lr'}. Best is trial 2 with value: 0.33066565035252565.
[I 2026-02-20 17:36:15,499] Trial 3 finished with value: 0.33066565035252565 and parameters: {'final_estimator': 'lr'}. Best is trial 2 with value: 0.33066565035252565.
[I 2026-02-20 17:37:45,086] Trial 4 finished with value: 0.5391491804682785 and parameters: {'final_estima

### Log the model and each model's metrics and parameters:

In [38]:
with mlflow.start_run(run_name="stacking_regressor"):


    # log child runs:
    for trial in study.trials:
        with mlflow.start_run(nested=True):

            # Log parameters:
            mlflow.log_params(trial.params)

            # log metrics:
            mlflow.log_metric("mean_absolute_error", trial.values[0])
            mlflow.log_metric("r2_score", trial.user_attrs["r2"])



    # create and train the final model:
    regressor = StackingRegressor(estimators=estimators,
                      final_estimator=LinearRegression(),
                      cv=5, n_jobs=-1)

    regressor.fit(X_train, y_train)
    y_train_pred = regressor.predict(X_train)
    y_pred = regressor.predict(X_test)

    # evaluate model:
    scores = cross_val_score(regressor, X_train,
                              y_train, cv=5,
                              scoring="neg_mean_absolute_error")
    metrics = {"mae_train": mean_absolute_error(y_train, y_train_pred),
               "r2_train": r2_score(y_train, y_train_pred),
               "mae_test": mean_absolute_error(y_test, y_pred),
               "r2_test": r2_score(y_test, y_pred),
               "neg_mean_absolute_error": scores.mean()}

    # log best parameters:
    mlflow.log_params(regressor.get_params())

    # log metrics:
    mlflow.log_metrics(metrics)

    # log model:
    signature = mlflow.models.infer_signature(X_test, y_pred)
    mlflow.sklearn.log_model(regressor, signature=signature)


🏃 View run handsome-mole-811 at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8/runs/32e001d5a13440d289dfd3dfa3316733
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8
🏃 View run indecisive-chimp-833 at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8/runs/029525803f164b1ab1b23112970b07ea
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8
🏃 View run sneaky-stoat-951 at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8/runs/c335904904cf41c8a24db92eedd990bf
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8
🏃 View run colorful-mouse-185 at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8/runs/9553e91c123b4bbc98fcaf606ecec540
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/20 18:37:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during de

🏃 View run stacking_regressor at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8/runs/33ee2afb195544f096848c6e0d1aad2e
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/8
